# Unificación de bases de datos de rasgos funcionales

Este cuaderno concatena y homogeniza las 5 bases de datos de rasgos funcionales
provenientes de distintos sitios/parcelas:

1. **Galeras** (`Rasgos_Galeras_unificado.xlsx`)
2. **Oyacachi–Guacamayos** (`Rasgos_Oyacachi_Guacamayos_unificado.xlsx`)
3. **Selva Viva** (`Rasgos_SelvaViva_unificado.xlsx`)
4. **Sumaco 2025** (`Rasgos_2025_con_rasgos_calculados.xlsx`)
5. **San Francisco (SF)** (`SFtraits_complete.xlsx`) — **solo se usan los controles**
   (columna `treat` / **SUB = "C"**)
6. **Sur – control histórico** (`Rasgos_south__control.xlsx`, hoja `hoja2`) — Bombuscaro,
   Cajanuma y San Francisco, años 2008–2023. Sus `treeID` que **no existen** en las otras
   bases se resaltan en **amarillo** en el Excel final.

El objetivo es:
- Homogeneizar los nombres de columnas (cada base trae nombres distintos para el mismo rasgo).
- Conservar la trazabilidad del origen de cada fila (`dataset_source`, `source_file`).
- Concatenar todo en una única tabla larga (`df_unificado`).
- Exportar el resultado a Excel y CSV.


## 1. Importar librerías y definir rutas

In [44]:
# ============================================================
# PASO 1: Importar las librerías que vamos a usar
# ============================================================



import pandas as pd        # pandas: para leer y manejar tablas de datos
import numpy as np         # numpy: para manejar valores vacíos (NaN) y cálculos
from pathlib import Path   # Path: para manejar rutas de archivos de forma segura

# Mostrar todas las columnas al imprimir una tabla (para que no se corte la vista)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ============================================================
# PASO 2: Definir las rutas de los archivos originales
# ============================================================
# OJO: hay que envolver cada ruta con Path(r"...") para que Python la
# reconozca como una "ruta de archivo" y no como simple texto.
# Así podemos usar cosas como .name (nombre del archivo) más adelante.
# C:\Users\Andrea\Downloads\NAPO_preliminar\2.RASGOS\BASE_actualizada.xlsx
FILES = {
    "Galeras": Path(r"C:\Users\Andrea\Downloads\NAPO_preliminar\Galeras\Rasgos_Galeras_unificado.xlsx"),
    "Oyacachi_Guacamayos": Path(r"C:\Users\Andrea\Downloads\NAPO_preliminar\Oyacachi-Guacamayos\Rasgos_Oyacachi_Guacamayos_unificado.xlsx"),
    "SelvaViva": Path(r"C:\Users\Andrea\Downloads\NAPO_preliminar\Selva Viva\Rasgos_SelvaViva_unificado.xlsx"),
    "Sumaco_2025": Path(r"C:\Users\Andrea\Downloads\NAPO_preliminar\Sumaco\Rasgos_2025_con_rasgos_calculados.xlsx"),
    "SF": Path(r"C:\Users\Andrea\Downloads\NAPO_preliminar\SF_control\data_for_R.xlsx"),
    # NUEVO: base histórica del sur (revisa que la ruta sea la correcta)
    "Sur_control": Path(r"C:\Users\Andrea\Downloads\NAPO_preliminar\2.RASGOS\Rasgos south  control.xlsx"),
}

# Carpeta donde vamos a guardar el resultado final (la unificada).
# Aquí uso la misma carpeta "NAPO_preliminar", pero puedes cambiarla.
OUT_DIR = Path(r"C:\Users\Andrea\Downloads\NAPO_preliminar")

# Nombre de la hoja (pestaña) de Excel que hay que leer en cada archivo.
# (Se ignora la hoja "Filas_a_revisar" porque es solo una bitácora de
# control de calidad, no datos de rasgos).
SHEETS = {
    "Galeras": "Rasgos_Galeras_unificado",
    "Oyacachi_Guacamayos": "Rasgos_OYA_GUA_unificado",
    "SelvaViva": "Rasgos_SelvaViva_unificado",
    "Sumaco_2025": "Rasgos_2025_calculado",
    "SF": "Sheet1",
    "Sur_control": "hoja2",      # NUEVO (la hoja "common spec overview" es solo un resumen)
}

# ============================================================
# PASO 3: Cargar cada archivo de Excel en un diccionario
# ============================================================
raw = {}  # aquí vamos a guardar cada tabla, usando el nombre del sitio como clave

for name, path in FILES.items():
    # Leer el archivo Excel, indicando la hoja correcta
    raw[name] = pd.read_excel(path, sheet_name=SHEETS[name])
    # Imprimir cuántas filas y columnas tiene cada tabla, y el nombre del archivo
    print(f"{name:22s} -> {raw[name].shape[0]:4d} filas, {raw[name].shape[1]:3d} columnas   ({path.name})")

Galeras                ->   90 filas,  44 columnas   (Rasgos_Galeras_unificado.xlsx)
Oyacachi_Guacamayos    ->   61 filas,  30 columnas   (Rasgos_Oyacachi_Guacamayos_unificado.xlsx)
SelvaViva              ->   62 filas,  60 columnas   (Rasgos_SelvaViva_unificado.xlsx)
Sumaco_2025            ->   49 filas,  51 columnas   (Rasgos_2025_con_rasgos_calculados.xlsx)
SF                     ->  174 filas,  67 columnas   (data_for_R.xlsx)
Sur_control            ->  585 filas,  18 columnas   (Rasgos south  control.xlsx)


## 3. Esquema homogeneizado

Se define un **esquema común** de columnas al que se mapean los nombres originales
de cada base. Cuando un rasgo no existe en una base, la columna queda como `NaN`.

Notas de homogeneización:

- `Subplot` en la base **SF** corresponde a la columna `treat` (tratamiento del
  experimento de fertilización). Como se pidió, **de SF solo se usan los controles
  (`SUB` = `"C"`)**.
- `SLA_cm2_g` (área foliar específica) no viene calculada en `Sumaco_2025` ni en `SF`;
  se calcula aquí como `leaf_area_cm2 / leaf_dry_weight_g`, igual que el criterio
  usado en las demás bases (columna `SLA_original`).
- Los elementos foliares (`C, N, P, Al, Ca, Fe, K, Mg, Na, S` en mg/g) solo existen
  en la base **SF**; en el resto quedan como `NaN`.
- `PlotID` de SF no viene en el archivo original, se construye como `"SF_" + Plot`.
- `genus` / `species` de SF se obtienen separando la columna `Species`
  (p. ej. `"Alchornea lojaensis"` -> genus=`Alchornea`, species=`lojaensis`).


## 4. Columnas del esquema final

In [45]:
CORE_COLUMNS = [
    "dataset_source", "Site", "PlotID", "Plot", "Subplot",
    "treeID", "new_tree_ID_2025", "family", "genus", "species",
    "sampling_date", "altitude_m",
    "n_leaves", "leaf_fresh_weight_g", "leaf_dry_weight_g", "leaf_area_cm2",
    "SLA_cm2_g", "LDMC_mg_g", "mean_leaf_thickness_mm",
    "wood_density_g_cm3", "WSG", "stem_water_content_pct", "force_to_punch_kN_m",
    "C_mg_g", "N_mg_g", "P_mg_g", "Al_mg_g", "Ca_mg_g",
    "Fe_mg_g", "K_mg_g", "Mg_mg_g", "Na_mg_g", "S_mg_g",
    "comment", "QC_flag", "source_file",
]

def empty_core_df(n):
    '''Crea un DataFrame vacio (NaN) con el esquema homogeneizado y n filas.'''
    return pd.DataFrame({c: [np.nan] * n for c in CORE_COLUMNS})


## 5. Homogeneizar Galeras

In [46]:
def standardize_galeras(df):
    out = empty_core_df(len(df))
    out["dataset_source"]      = "Galeras"
    out["Site"]                = df["Site"]
    out["PlotID"]              = df["PlotID"]
    out["Plot"]                = df["Plot"]
    out["Subplot"]             = df["Subplot"]
    out["treeID"]              = df["old_ID"]
    out["new_tree_ID_2025"]              = df["new_ID"]
    out["family"]              = df["family"]
    out["genus"]               = df["genus"]
    out["species"]             = df["species"]
    out["sampling_date"]       = df["sampling_date"]
    out["altitude_m"]          = df["altitude_m"]
    out["n_leaves"]            = df["n_leaves"]
    out["leaf_dry_weight_g"]   = df["leaf_dry_weight_g"]
    out["leaf_area_cm2"]       = df["leaf_area_cm2"]
    out["SLA_cm2_g"]           = df["SLA_cm2_g"]
    out["LDMC_mg_g"]           = df["LDMC_mg_g"]
    out["mean_leaf_thickness_mm"] = df["mean_leaf_thickness_mm"]
    out["wood_density_g_cm3"]  = df["wood_density_g_cm3"]
    out["WSG"]                 = df["WSG"]
    out["stem_water_content_pct"] = df["stem_water_content_pct"]
    out["force_to_punch_kN_m"] = df["force_to_punch_kN_m"]
    out["comment"]             = df["comment"]
    out["QC_flag"]             = df["QC_flag"]
    out["source_file"]         = df["source_file"]
    return out

df_galeras = standardize_galeras(raw["Galeras"])
df_galeras.head(3)


,dataset_source,Site,PlotID,Plot,Subplot,treeID,new_tree_ID_2025,family,genus,species,sampling_date,altitude_m,n_leaves,leaf_fresh_weight_g,leaf_dry_weight_g,leaf_area_cm2,SLA_cm2_g,LDMC_mg_g,mean_leaf_thickness_mm,wood_density_g_cm3,WSG,stem_water_content_pct,force_to_punch_kN_m,C_mg_g,N_mg_g,P_mg_g,Al_mg_g,Ca_mg_g,Fe_mg_g,K_mg_g,Mg_mg_g,Na_mg_g,S_mg_g,comment,QC_flag,source_file
0,Galeras,Galeras,GAL_24,24,A,590.0,8633,Clusiaceae,Garcinia,madruno,2025-01-08 00:00:00,1450.0,20.0,NaN,17.47,1568.870,89.803663,NaN,0.218000,0.684366,0.684366,93.023256,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,old_ID asignado por taxonomía en el plot (veri...,03__Rasgos_Galeras.xlsx (principal)
1,Galeras,Galeras,GAL_24,24,B,NaN,8639,Lacistemataceae,Lozania,klugii,2025-01-08 00:00:00,1450.0,20.0,NaN,10.22,1594.462,156.013894,NaN,0.089000,0.528159,0.528159,140.476190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sin old_ID;,03__Rasgos_Galeras.xlsx (principal)
2,Galeras,Galeras,GAL_24,24,B,NaN,8640,indet,NaN,NaN,2025-01-08 00:00:00,1450.0,20.0,NaN,9.03,645.064,71.435659,NaN,0.151111,0.396119,0.396119,197.619048,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mas 10 Nutrientes,sin old_ID;,03__Rasgos_Galeras.xlsx (principal)


## 6. Homogeneizar Oyacachi–Guacamayos

In [47]:
def standardize_oya_gua(df):
    out = empty_core_df(len(df))
    out["dataset_source"]      = "Oyacachi_Guacamayos"
    out["Site"]                = df["Site"]
    out["PlotID"]              = df["PlotID"]
    out["Plot"]                = df["Plot"]
    out["Subplot"]             = df["Subplot"]
    out["treeID"]              = df["treeID"]
    out["new_tree_ID_2025"]    = df["new_tree_ID_2025"]
    out["family"]              = df["family"]
    out["genus"]               = df["genus"]
    out["species"]             = df["species"]
    out["sampling_date"]       = df["sampling_date"]
    out["altitude_m"]          = df["altitude_m"]
    out["n_leaves"]            = df["n_leaves"]
    out["leaf_fresh_weight_g"] = df["leaf_fresh_weight_g"]
    out["leaf_dry_weight_g"]   = df["leaf_dry_weight_g"]
    out["leaf_area_cm2"]       = df["leaf_area_cm2"]
    out["SLA_cm2_g"]           = df["SLA_cm2_g"]
    out["LDMC_mg_g"]           = df["LDMC_mg_g"]
    out["mean_leaf_thickness_mm"] = df["mean_leaf_thickness_mm"]
    out["wood_density_g_cm3"]  = df["wood_density_g_cm3"]
    out["WSG"]                 = df["WSG"]
    out["stem_water_content_pct"] = df["stem_water_content_pct"]
    out["force_to_punch_kN_m"] = df["force_to_punch_kN_m"]
    out["comment"]             = df["comment"]
    out["QC_flag"]             = df["QC_flag"]
    out["source_file"]         = df["source_file"]
    return out

df_oya_gua = standardize_oya_gua(raw["Oyacachi_Guacamayos"])
df_oya_gua.head(3)


,dataset_source,Site,PlotID,Plot,Subplot,treeID,new_tree_ID_2025,family,genus,species,sampling_date,altitude_m,n_leaves,leaf_fresh_weight_g,leaf_dry_weight_g,leaf_area_cm2,SLA_cm2_g,LDMC_mg_g,mean_leaf_thickness_mm,wood_density_g_cm3,WSG,stem_water_content_pct,force_to_punch_kN_m,C_mg_g,N_mg_g,P_mg_g,Al_mg_g,Ca_mg_g,Fe_mg_g,K_mg_g,Mg_mg_g,Na_mg_g,S_mg_g,comment,QC_flag,source_file
0,Oyacachi_Guacamayos,Oyacachi,OYC-81,81,NaN,4020.0,8016,Rosaceae,Polylepis,pauta,2023-10-20,3978,7,NaN,0.62,56.078069,90.448498,NaN,0.235333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01__Rasgos_Oyacachi.xlsx
1,Oyacachi_Guacamayos,Oyacachi,OYC-81,81,NaN,4021.0,8013,Asteraceae,Gynoxis,acostae,2023-10-20,3978,10,NaN,4.07,282.222341,69.342099,NaN,0.506333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01__Rasgos_Oyacachi.xlsx
2,Oyacachi_Guacamayos,Oyacachi,OYC-81,81,NaN,4022.0,8039,Rosaceae,Polylepis,pauta,2023-10-20,3978,10,NaN,1.55,121.997778,78.708244,NaN,0.247000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,01__Rasgos_Oyacachi.xlsx


## 7. Homogeneizar Selva Viva

In [48]:
def standardize_selvaviva(df):
    out = empty_core_df(len(df))
    out["dataset_source"]      = "SelvaViva"
    out["Site"]                = df["Site"]
    out["PlotID"]              = df["PlotID"]
    out["Plot"]                = df.get("Plot_x", np.nan)
    out["Subplot"]             = df["Subplot"]
    out["treeID"]              = df["treeID"]
    out["new_tree_ID_2025"]    = df["new_tree_ID_2025"]
    out["family"]              = df["family"]
    out["genus"]               = df["genus"]
    out["species"]             = df["species"]
    out["sampling_date"]       = df["sampling_date"]
    out["altitude_m"]          = df["altitude_m"]
    out["n_leaves"]            = df["n_leaves"]
    out["leaf_fresh_weight_g"] = df["leaf_fresh_weight_g"]
    out["leaf_dry_weight_g"]   = df["leaf_dry_weight_g"]
    out["leaf_area_cm2"]       = df["leaf_area_cm2"]
    out["SLA_cm2_g"]           = df["SLA_cm2_g"]
    out["LDMC_mg_g"]           = df["LDMC_mg_g"]
    out["mean_leaf_thickness_mm"] = df["mean_leaf_thickness_mm"]
    out["wood_density_g_cm3"]  = df["wood_density_g_cm3"]
    out["WSG"]                 = df["WSG"]
    out["stem_water_content_pct"] = df["stem_water_content_pct"]
    out["force_to_punch_kN_m"] = df["force_to_punch_kN_m"]
    out["comment"]             = df["comment"]
    out["QC_flag"]             = df["QC_flag"]
    out["source_file"]         = df["source_file"]
    return out

df_selvaviva = standardize_selvaviva(raw["SelvaViva"])
df_selvaviva.head(3)


,dataset_source,Site,PlotID,Plot,Subplot,treeID,new_tree_ID_2025,family,genus,species,sampling_date,altitude_m,n_leaves,leaf_fresh_weight_g,leaf_dry_weight_g,leaf_area_cm2,SLA_cm2_g,LDMC_mg_g,mean_leaf_thickness_mm,wood_density_g_cm3,WSG,stem_water_content_pct,force_to_punch_kN_m,C_mg_g,N_mg_g,P_mg_g,Al_mg_g,Ca_mg_g,Fe_mg_g,K_mg_g,Mg_mg_g,Na_mg_g,S_mg_g,comment,QC_flag,source_file
0,SelvaViva,Selva Viva,SEV_69,69,B,NaN,8898,Euphorbiaceae,Pseudosenefeldera,inclinata,2025-02-04,508.0,15.0,53.68,24.61,2418.427,98.270093,458.457526,0.145556,0.819648,0.819648,63.106796,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"04__Rasgos_SelvaViva.xlsx (principal, feb-2025)"
1,SelvaViva,Selva Viva,SEV_69,69,A,NaN,8891,Sapindaceae,Allophylus,punctatus,2025-02-04,508.0,15.0,29.80,10.21,2005.277,196.403232,342.617450,0.191111,0.509296,0.509296,117.391304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"04__Rasgos_SelvaViva.xlsx (principal, feb-2025)"
2,SelvaViva,Selva Viva,SEV_69,69,A,NaN,8892,Rubiaceae,Pentagonia,NaN,2025-02-04,508.0,15.0,92.61,22.77,2509.199,110.197585,245.869776,0.226667,0.518390,0.518390,136.842105,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"04__Rasgos_SelvaViva.xlsx (principal, feb-2025)"


## 8. Homogeneizar Sumaco 2025

In [49]:
def standardize_sumaco2025(df):
    out = empty_core_df(len(df))
    out["dataset_source"]      = "Sumaco_2025"
    out["Site"]                = "Sumaco"
    out["PlotID"]              = df["plot ID"]
    out["Plot"]                = df["Plot"]
    out["treeID"]              = df["treeID"]
    out["new_tree_ID_2025"]    = df["new tree ID 2025"]
    out["family"]              = df["family"]
    out["genus"]               = df["genus"]
    out["species"]             = df["species"]
    out["sampling_date"]       = df["sampling date_leaves 2025"]
    out["n_leaves"]            = df["No. leaves"]
    out["leaf_fresh_weight_g"] = df["Leaf fresh weight (g)"]
    out["leaf_dry_weight_g"]   = df["Leaf dry weight (g)"]
    out["leaf_area_cm2"]       = df["Leaf area (cm\u00b2)"]
    # SLA no viene calculada en esta base -> se calcula igual que en las demas
    out["SLA_cm2_g"]           = df["Leaf area (cm\u00b2)"] / df["Leaf dry weight (g)"]
    out["LDMC_mg_g"]           = df["LDMC_mg_g"]
    out["mean_leaf_thickness_mm"] = df["mean_leaf_thickness_mm"]
    out["wood_density_g_cm3"]  = df["wood_density_g_cm3"]
    out["WSG"]                 = df["WSG"]
    out["stem_water_content_pct"] = df["stem_water_content_pct"]
    # combinar las dos columnas de comentarios en una sola
    out["comment"] = (
        df["comment"].fillna("").astype(str) + " | " + df["Comments 2025"].fillna("").astype(str)
    ).str.strip(" |").replace("", np.nan)
    out["QC_flag"]             = df["QC_flag_rasgos"]
    out["source_file"]         = "Rasgos_2025_con_rasgos_calculados.xlsx"
    return out

df_sumaco2025 = standardize_sumaco2025(raw["Sumaco_2025"])
df_sumaco2025.head(3)


,dataset_source,Site,PlotID,Plot,Subplot,treeID,new_tree_ID_2025,family,genus,species,sampling_date,altitude_m,n_leaves,leaf_fresh_weight_g,leaf_dry_weight_g,leaf_area_cm2,SLA_cm2_g,LDMC_mg_g,mean_leaf_thickness_mm,wood_density_g_cm3,WSG,stem_water_content_pct,force_to_punch_kN_m,C_mg_g,N_mg_g,P_mg_g,Al_mg_g,Ca_mg_g,Fe_mg_g,K_mg_g,Mg_mg_g,Na_mg_g,S_mg_g,comment,QC_flag,source_file
0,Sumaco_2025,Sumaco,SUM_09,9,NaN,233,3730,Metteniusaceae,Calatola,costaricensis,2025-09-02,NaN,15,34.91,7.45,1434.898,192.603758,213.405901,0.208167,0.573628,0.573628,120.560748,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rasgos_2025_con_rasgos_calculados.xlsx
1,Sumaco_2025,Sumaco,SUM_09,9,NaN,235,3733,Metteniusaceae,Calatola,costaricensis,2025-09-02,NaN,20,58.20,14.14,1747.557,123.589604,242.955326,0.228167,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rasgos_2025_con_rasgos_calculados.xlsx
2,Sumaco_2025,Sumaco,SUM_09,9,NaN,NaN,3735,Sapindaceae,Allophylus,sp,2025-09-02,NaN,10,58.26,21.59,3603.677,166.914173,370.580158,0.245000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rasgos_2025_con_rasgos_calculados.xlsx


## 9. Homogeneizar SF (San Francisco) — solo controles (`SUB = "C"`)

Se filtra primero la base para conservar **únicamente las filas de control**
(`treat == "C"`), que es el subplot de control del experimento de fertilización.


In [50]:
def standardize_sf(df):
    # --- limpiar espacios en blanco en la columna Site ---
    df = df.copy()
    df["Site"] = df["Site"].astype(str).str.strip()

    # --- filtrar solo los controles: Sub = "C" ---
    df = df[df["Sub"] == "C"].copy()

    # *** LA LÍNEA CLAVE QUE FALTABA ***
    # Reinicia el índice (0,1,2,3...) para que coincida con el índice
    # de la tabla "out" que vamos a crear a continuación.
    df = df.reset_index(drop=True)

    # crear la tabla vacía con el esquema homogeneizado, del mismo largo que df
    out = empty_core_df(len(df))

    out["dataset_source"]      = "SF"
    out["Site"]                = df["Site"]
    out["PlotID"]              = df["Site"] + "_" + df["Plot"].astype(str)
    out["Plot"]                = df["Plot"]
    out["Subplot"]             = df["Sub"]
    out["treeID"]              = df["TreeID"]
    out["family"]              = np.nan
    out["genus"]               = df["genus"]
    out["species"]             = df["specie"]
    out["sampling_date"]       = df["Sampling date"]
    out["altitude_m"]          = df["elevation_m"]
    out["n_leaves"]            = df["number of leaves"]
    out["leaf_fresh_weight_g"] = df["Leaf fresh weight (g)"]
    out["leaf_dry_weight_g"]   = df["Leaf dry weight (g)"]
    out["leaf_area_cm2"]       = df["Leaf area (cm\u00b2)"]
    out["SLA_cm2_g"]           = df["Specific leaf area (cm\u00b2/g)"]
    out["LDMC_mg_g"]           = df["dry matter content (mg/g)"]
    out["mean_leaf_thickness_mm"] = df["thickness (mm)"]
    out["wood_density_g_cm3"]  = df["WD"]
    out["WSG"]                 = df["WD"]
    out["force_to_punch_kN_m"] = df["force to punch (kN/m)"]

    out["C_mg_g"]  = df["C (mg/g)"]
    out["N_mg_g"]  = df["N (mg/g)"]
    out["P_mg_g"]  = df["P (mg/g)"]
    out["Al_mg_g"] = df["Al (mg/g)"]
    out["Ca_mg_g"] = df["Ca (mg/g)"]
    out["Fe_mg_g"] = df["Fe (mg/g)"]
    out["K_mg_g"]  = df["K (mg/g)"]
    out["Mg_mg_g"] = df["Mg (mg/g)"]
    out["Na_mg_g"] = df["Na (mg/g)"]
    out["S_mg_g"]  = df["S (mg/g)"]

    out["source_file"] = "data_for_R.xlsx"
    return out

df_sf = standardize_sf(raw["SF"])
print(f"SF: {raw['SF'].shape[0]} filas originales -> {df_sf.shape[0]} filas de control (Sub = 'C')")
print(df_sf["Site"].value_counts())
df_sf.head()


SF: 174 filas originales -> 49 filas de control (Sub = 'C')
Site
SFNu    27
BoN     12
CaNu    10
Name: count, dtype: int64


,dataset_source,Site,PlotID,Plot,Subplot,treeID,new_tree_ID_2025,family,genus,species,sampling_date,altitude_m,n_leaves,leaf_fresh_weight_g,leaf_dry_weight_g,leaf_area_cm2,SLA_cm2_g,LDMC_mg_g,mean_leaf_thickness_mm,wood_density_g_cm3,WSG,stem_water_content_pct,force_to_punch_kN_m,C_mg_g,N_mg_g,P_mg_g,Al_mg_g,Ca_mg_g,Fe_mg_g,K_mg_g,Mg_mg_g,Na_mg_g,S_mg_g,comment,QC_flag,source_file
0,SF,CaNu,CaNu_1,1,C,1608,NaN,NaN,Weinmania,loxensis,2024-12-04,3000,20,0.69,0.34,24.302783,71.478774,492.753623,0.256500,0.536,0.536,NaN,0.770311,NaN,NaN,0.586049,0.308239,5.338353,0.051383,3.574373,3.923072,0.119075,1.517739,NaN,NaN,data_for_R.xlsx
1,SF,CaNu,CaNu_1,1,C,5861,NaN,NaN,Weinmania,loxensis,2024-12-04,3000,20,0.76,0.36,26.330000,73.138889,473.684211,0.234000,0.536,0.536,NaN,0.746437,NaN,NaN,0.586801,0.339795,6.484631,0.071215,4.302546,3.726822,0.033105,1.089162,NaN,NaN,data_for_R.xlsx
2,SF,BoN,BoN_4,4,C,2048,NaN,NaN,Pouteria,torta,2024-11-18,1000,20,46.05,23.87,1799.580000,75.390867,518.349620,0.201167,0.788,0.788,NaN,0.970581,NaN,NaN,0.733080,0.000000,2.158710,0.034950,5.827832,0.830238,0.079408,1.277097,NaN,NaN,data_for_R.xlsx
3,SF,BoN,BoN_4,4,C,2060,NaN,NaN,Pouteria,torta,2024-11-18,1000,20,35.63,16.73,1623.664000,97.051046,469.548134,0.190500,0.788,0.788,NaN,0.728930,NaN,NaN,0.692635,0.000000,1.755878,0.057200,5.931828,0.900499,1.190908,1.823890,NaN,NaN,data_for_R.xlsx
4,SF,BoN,BoN_4,4,C,2062,NaN,NaN,Pouteria,torta,2024-11-21,1000,20,34.11,18.97,1362.462000,71.821929,556.141894,0.201667,0.788,0.788,NaN,0.897369,NaN,NaN,0.549719,0.000000,1.666264,0.040715,4.758459,0.613314,0.409118,1.430080,NaN,NaN,data_for_R.xlsx


## 9b. Homogeneizar Sur – control histórico (`Rasgos_south__control.xlsx`)

Base larga: **una fila por árbol y por año** (`study_year`, 2008–2023), solo controles (`treat = "C"`).

Qué se hace:
- Los sitios `Bo`, `Ca`, `SF` se renombran a `BoN`, `CaNu`, `SFNu` para que coincidan con la base **SF**
  y se puedan comparar los `treeID`.
- `species` viene como "Género especie" → se separa en `genus` y `species`.
- `new treeID` se anotó solo en algunos años → se copia a **todos los años del mismo árbol**.
- Textos en columnas de ID o de rasgos (`no ID`, `al lado de #1487`, `soon come`…) se pasan a `comment`.
- `tough` se guarda en `force_to_punch_kN_m` porque los valores están en el mismo rango que SF
  (0.5–1.0). **Confirma las unidades.**
- Se agrega la columna `year` (las otras bases la toman de `sampling_date`).
- Se quitan las filas sin ningún rasgo medido (solo tienen el árbol y el año).

In [51]:
# Nombres de sitio del sur -> mismos nombres que en la base SF
SITIOS_SUR = {"Bo": "BoN", "Ca": "CaNu", "SF": "SFNu"}


def id_a_numero(serie):
    """Convierte IDs a número. Lo que no es número (texto) queda vacío."""
    return pd.to_numeric(serie, errors="coerce")


def standardize_sur(df):
    df = df.copy()
    df.columns = df.columns.str.strip()          # la columna 'SLA ' trae un espacio al final
    df = df.reset_index(drop=True)
    comentario = pd.Series("", index=df.index)   # aquí se van juntando las notas de cada fila

    # --- treeID antiguo -------------------------------------------------------
    # 'al lado de #1487' y 'alado 1487' son el mismo árbol -> 'al_lado_1487'
    old = df["treeID"].astype(str).str.strip()
    es_lado = old.str.contains("lado", case=False)
    old[es_lado] = "al_lado_" + old[es_lado].str.extract(r"(\d+)")[0]
    old_num = id_a_numero(old)
    tree_id = old_num.astype(object).where(old_num.notna(), old)   # número si se puede, si no texto
    tree_id[df["treeID"].isna()] = np.nan

    # --- new treeID -------------------------------------------------------------
    # 'no ID', 'A lado de 1265' -> vacío ; '4966, 9683' -> 4966 ; el texto original va a comment
    new_txt = df["new treeID"].astype("string")
    new_id = id_a_numero(df["new treeID"])
    con_texto = new_id.isna() & df["new treeID"].notna()
    comentario[con_texto] += "new treeID original: " + new_txt[con_texto] + "; "
    dos_ids = new_txt.str.contains(",", na=False)
    new_id[dos_ids] = id_a_numero(new_txt[dos_ids].str.extract(r"(\d+)")[0])

    # Copiar el new treeID a todos los años del mismo árbol (sitio + plot + treeID)
    arbol = [df["site"], df["plot"], tree_id.astype(str)]
    n_distintos = new_id.groupby(arbol).transform("nunique")
    new_id = new_id.groupby(arbol).transform("first").where(n_distintos == 1, new_id)
    comentario[n_distintos > 1] += "árbol con varios new treeID entre años; "

    # --- N con texto ('soon come') ------------------------------------------------
    n_num = pd.to_numeric(df["N"], errors="coerce")
    texto_n = n_num.isna() & df["N"].notna()
    comentario[texto_n] += "N: " + df.loc[texto_n, "N"].astype(str) + "; "

    # --- Tabla con el esquema común ------------------------------------------------
    out = empty_core_df(len(df))
    out["dataset_source"]   = "Sur_control"
    out["Site"]             = df["site"].map(SITIOS_SUR).fillna(df["site"])
    out["PlotID"]           = out["Site"] + "_" + df["plot"].astype(str)      # igual que SF: BoN_4
    out["Plot"]             = df["plot"]
    out["Subplot"]          = df["treat"]                                     # "C" = control
    out["treeID"]           = tree_id
    out["new_tree_ID_2025"] = new_id
    partes = df["species"].str.strip().str.split(n=1, expand=True)
    out["genus"]            = partes[0]
    out["species"]          = partes[1]
    out["sampling_date"]    = df["sampling date"]
    out["leaf_area_cm2"]    = df["LA"]
    out["SLA_cm2_g"]        = df["SLA"]
    out["mean_leaf_thickness_mm"] = df["thick"]
    out["force_to_punch_kN_m"]    = df["tough"]        # confirmar unidades
    out["N_mg_g"]           = n_num
    out["P_mg_g"]           = df["P"]
    out["comment"]          = comentario.replace("", np.nan)
    out["source_file"]      = "Rasgos_south__control.xlsx"
    out["year"]             = df["study_year"]           # columna nueva (no está en CORE_COLUMNS)

    # --- Quitar filas sin ningún rasgo -------------------------------------------
    rasgos = ["leaf_area_cm2", "SLA_cm2_g", "mean_leaf_thickness_mm", "force_to_punch_kN_m", "N_mg_g", "P_mg_g"]
    sin_rasgos = out[rasgos].isna().all(axis=1)
    print(f"Sur: {len(out)} filas -> se quitan {sin_rasgos.sum()} sin ningún rasgo")
    return out[~sin_rasgos].reset_index(drop=True)


df_sur = standardize_sur(raw["Sur_control"])
print(f"Sur: {len(df_sur)} filas, {df_sur.groupby(['Site', 'Plot', 'treeID']).ngroups} árboles")
print(df_sur["Site"].value_counts())
df_sur.head()

Sur: 585 filas -> se quitan 109 sin ningún rasgo
Sur: 476 filas, 116 árboles
Site
SFNu    276
BoN     102
CaNu     98
Name: count, dtype: int64


,dataset_source,Site,PlotID,Plot,Subplot,treeID,new_tree_ID_2025,family,genus,species,sampling_date,altitude_m,n_leaves,leaf_fresh_weight_g,leaf_dry_weight_g,leaf_area_cm2,SLA_cm2_g,LDMC_mg_g,mean_leaf_thickness_mm,wood_density_g_cm3,WSG,stem_water_content_pct,force_to_punch_kN_m,C_mg_g,N_mg_g,P_mg_g,Al_mg_g,Ca_mg_g,Fe_mg_g,K_mg_g,Mg_mg_g,Na_mg_g,S_mg_g,comment,QC_flag,source_file,year
0,Sur_control,BoN,BoN_8,8,C,1917.0,NaN,NaN,Clarisia,racemosa,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.28773,0.849073,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rasgos_south__control.xlsx,2017
1,Sur_control,BoN,BoN_8,8,C,1930.0,NaN,NaN,Clarisia,racemosa,NaT,NaN,NaN,NaN,NaN,23.776450,89.876615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.47409,0.558062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rasgos_south__control.xlsx,2013
2,Sur_control,BoN,BoN_8,8,C,1930.0,NaN,NaN,Clarisia,racemosa,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.24450,0.736694,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rasgos_south__control.xlsx,2014
3,Sur_control,BoN,BoN_8,8,C,1930.0,NaN,NaN,Clarisia,racemosa,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.83492,0.842525,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rasgos_south__control.xlsx,2017
4,Sur_control,BoN,BoN_8,8,C,1930.0,NaN,NaN,Clarisia,racemosa,NaT,NaN,NaN,NaN,NaN,24.007061,101.314202,NaN,NaN,NaN,NaN,NaN,0.905061,NaN,16.64191,0.560221,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rasgos_south__control.xlsx,2018


## 9c. ¿Los `treeID` del sur ya existen en las otras bases?

Se compara **sitio + ID**, porque el mismo número puede repetirse en sitios distintos.
En la base SF a veces el ID nuevo está en la columna `treeID` (p. ej. 5861), por eso se juntan
`treeID` y `new_tree_ID_2025` de las otras bases y se busca ahí tanto el `treeID` como el
`new_tree_ID_2025` del sur.

Resultado: columna `existe_en_otras_bases` (`True` / `False`). Los `False` se pintan de amarillo al guardar.

In [52]:
def id_texto(serie):
    """IDs como texto comparable: 370, 370.0 y '370' -> '370'. Vacíos quedan vacíos."""
    num = pd.to_numeric(serie, errors="coerce")
    txt = num.round().astype("Int64").astype("string")
    return txt.fillna(serie.astype("string").str.strip())       # IDs de texto se dejan igual


def llave(df, col):
    """'Sitio|ID' para cada fila (vacío si no hay ID)."""
    return df["Site"].astype("string").str.strip() + "|" + id_texto(df[col])


# Todas las bases menos la del sur
otras = pd.concat([df_galeras, df_oya_gua, df_selvaviva, df_sumaco2025, df_sf], ignore_index=True)

# Conjunto con todos los 'Sitio|ID' que existen (treeID y new_tree_ID_2025 juntos)
ids_existentes = set(llave(otras, "treeID").dropna()) | set(llave(otras, "new_tree_ID_2025").dropna())

existe_old = llave(df_sur, "treeID").isin(ids_existentes).fillna(False).astype(bool)
existe_new = llave(df_sur, "new_tree_ID_2025").isin(ids_existentes).fillna(False).astype(bool)
df_sur["existe_en_otras_bases"] = existe_old | existe_new

print("Filas del sur cuyo árbol YA existe   :", df_sur["existe_en_otras_bases"].sum())
print("Filas del sur cuyo árbol NO existe   :", (~df_sur["existe_en_otras_bases"]).sum(), "-> se resaltan")

# Resumen por árbol, para revisar
(df_sur.groupby(["Site", "Plot", "treeID"], dropna=False)
       .agg(new_tree_ID_2025=("new_tree_ID_2025", "first"),
            especie=("genus", "first"),
            años=("year", lambda y: ", ".join(map(str, sorted(y.unique())))),
            existe=("existe_en_otras_bases", "first"))
       .reset_index()
       .sort_values(["existe", "Site", "Plot"]))

Filas del sur cuyo árbol YA existe   : 241
Filas del sur cuyo árbol NO existe   : 235 -> se resaltan


,Site,Plot,treeID,new_tree_ID_2025,especie,años,existe
1,BoN,4,2056.0,NaN,Pouteria,2018,False
4,BoN,8,1917.0,NaN,Clarisia,2017,False
5,BoN,8,1918.0,NaN,Pouteria,2018,False
10,BoN,8,1934.0,NaN,Clarisia,2018,False
12,BoN,8,1949.0,NaN,Pouteria,"2009, 2010",False
...,...,...,...,...,...,...,...
106,SFNu,17,882.0,4932.0,Alchornea,"2011, 2018, 2023",True
109,SFNu,17,889.0,7645.0,Alchornea,2023,True
111,SFNu,17,902.0,NaN,Myrcia,"2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015...",True
116,SFNu,17,4933.0,NaN,Myrcia,2023,True


## 10. Concatenar todas las bases homogeneizadas

In [53]:
df_unificado = pd.concat(
    [df_galeras, df_oya_gua, df_selvaviva, df_sumaco2025, df_sf, df_sur],   # NUEVO: df_sur
    ignore_index=True,
)

# tipos de datos consistentes
df_unificado["sampling_date"] = pd.to_datetime(df_unificado["sampling_date"], errors="coerce")

# NUEVO: año de muestreo para todas las bases (el sur ya lo trae; las demás lo sacan de la fecha)
df_unificado["year"] = df_unificado["year"].fillna(df_unificado["sampling_date"].dt.year).astype("Int64")

numeric_cols = [
    "n_leaves", "leaf_fresh_weight_g", "leaf_dry_weight_g", "leaf_area_cm2",
    "SLA_cm2_g", "LDMC_mg_g", "mean_leaf_thickness_mm", "wood_density_g_cm3", "WSG",
    "stem_water_content_pct", "force_to_punch_kN_m",
    "C_mg_g", "N_mg_g", "P_mg_g", "Al_mg_g", "Ca_mg_g", "Fe_mg_g", "K_mg_g", "Mg_mg_g", "Na_mg_g", "S_mg_g",
] #"altitude_m"
for c in numeric_cols:
    df_unificado[c] = pd.to_numeric(df_unificado[c], errors="coerce")

print(f"Base unificada: {df_unificado.shape[0]} filas x {df_unificado.shape[1]} columnas")
df_unificado.head()


Base unificada: 787 filas x 38 columnas


,dataset_source,Site,PlotID,Plot,Subplot,treeID,new_tree_ID_2025,family,genus,species,sampling_date,altitude_m,n_leaves,leaf_fresh_weight_g,leaf_dry_weight_g,leaf_area_cm2,SLA_cm2_g,LDMC_mg_g,mean_leaf_thickness_mm,wood_density_g_cm3,WSG,stem_water_content_pct,force_to_punch_kN_m,C_mg_g,N_mg_g,P_mg_g,Al_mg_g,Ca_mg_g,Fe_mg_g,K_mg_g,Mg_mg_g,Na_mg_g,S_mg_g,comment,QC_flag,source_file,year,existe_en_otras_bases
0,Galeras,Galeras,GAL_24,24,A,590.0,8633.0,Clusiaceae,Garcinia,madruno,2025-01-08,1450.0,20.0,NaN,17.47,1568.870,89.803663,NaN,0.218000,0.684366,0.684366,93.023256,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,old_ID asignado por taxonomía en el plot (veri...,03__Rasgos_Galeras.xlsx (principal),2025,NaN
1,Galeras,Galeras,GAL_24,24,B,NaN,8639.0,Lacistemataceae,Lozania,klugii,2025-01-08,1450.0,20.0,NaN,10.22,1594.462,156.013894,NaN,0.089000,0.528159,0.528159,140.476190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sin old_ID;,03__Rasgos_Galeras.xlsx (principal),2025,NaN
2,Galeras,Galeras,GAL_24,24,B,NaN,8640.0,indet,NaN,NaN,2025-01-08,1450.0,20.0,NaN,9.03,645.064,71.435659,NaN,0.151111,0.396119,0.396119,197.619048,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mas 10 Nutrientes,sin old_ID;,03__Rasgos_Galeras.xlsx (principal),2025,NaN
3,Galeras,Galeras,GAL_24,24,B,580.0,8641.0,Moraceae,Clarisia,biflora,2025-01-08,1450.0,20.0,NaN,11.94,1402.936,117.498827,NaN,0.176667,0.697666,0.697666,81.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,old_ID asignado por taxonomía en el plot (veri...,03__Rasgos_Galeras.xlsx (principal),2025,NaN
4,Galeras,Galeras,GAL_24,24,B,579.0,8642.0,Lecythidaceae,Eschweilera,caudiculata,2025-01-08,1450.0,20.0,NaN,5.06,541.494,107.014625,NaN,0.175556,0.754022,0.754022,78.947368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,old_ID asignado por taxonomía en el plot (veri...,03__Rasgos_Galeras.xlsx (principal),2025,NaN


## 11. Control de calidad: resumen por fuente

In [54]:
summary = df_unificado.groupby("dataset_source").agg(
    n_filas=("treeID", "size"),
    n_sitios=("Site", "nunique"),
    n_familias=("family", "nunique"),
    con_SLA=("SLA_cm2_g", lambda s: s.notna().sum()),
    con_LDMC=("LDMC_mg_g", lambda s: s.notna().sum()),
    con_densidad_madera=("wood_density_g_cm3", lambda s: s.notna().sum()),
).reset_index()
summary


,dataset_source,n_filas,n_sitios,n_familias,con_SLA,con_LDMC,con_densidad_madera
0,Galeras,90,1,29,77,0,80
1,Oyacachi_Guacamayos,61,2,14,61,3,0
2,SF,49,3,0,49,49,49
3,SelvaViva,62,1,25,59,59,57
4,Sumaco_2025,49,1,17,47,49,19
5,Sur_control,476,3,0,259,0,0


In [55]:
# porcentaje de valores faltantes por columna, para revisión rápida
missing_pct = (df_unificado.isna().mean() * 100).round(1).sort_values(ascending=False)
missing_pct.to_frame("pct_faltante")


,pct_faltante
C_mg_g,96.6
K_mg_g,93.8
S_mg_g,93.8
Na_mg_g,93.8
Ca_mg_g,93.8
Mg_mg_g,93.8
Al_mg_g,93.8
Fe_mg_g,93.8
QC_flag,91.9
comment,87.7


## 12. Guardar la base unificada

In [56]:
import openpyxl
from openpyxl.styles import PatternFill

out_xlsx = OUT_DIR / "Rasgos_unificado_todas_las_bases.xlsx"
#out_csv  = OUT_DIR / "Rasgos_unificado_todas_las_bases.csv"

df_unificado.to_excel(out_xlsx, index=False)
#df_unificado.to_csv(out_csv, index=False)

# ------------------------------------------------------------------
# NUEVO: resaltar en amarillo treeID y new_tree_ID_2025 de las filas
# del sur cuyo árbol NO existe en las otras bases
# ------------------------------------------------------------------
wb = openpyxl.load_workbook(out_xlsx)
ws = wb.active
columna = {celda.value: celda.column for celda in ws[1]}      # nombre de columna -> número
amarillo = PatternFill(start_color="FFFFFF00", end_color="FFFFFF00", fill_type="solid")

no_existe = df_unificado["existe_en_otras_bases"].eq(False)    # solo filas del sur con False
for i in np.where(no_existe)[0]:
    fila_excel = i + 2          # +1 por el encabezado, +1 porque Excel empieza en 1
    ws.cell(row=fila_excel, column=columna["treeID"]).fill = amarillo
    ws.cell(row=fila_excel, column=columna["new_tree_ID_2025"]).fill = amarillo

wb.save(out_xlsx)
print(f"Guardado: {out_xlsx}")
print(f"Celdas resaltadas (filas del sur sin coincidencia): {no_existe.sum()}")
#print(f"Guardado: {out_csv}")

Guardado: C:\Users\Andrea\Downloads\NAPO_preliminar\Rasgos_unificado_todas_las_bases.xlsx
Celdas resaltadas (filas del sur sin coincidencia): 235
